<!--
  SPDX-FileCopyrightText: 2026 Advanced Micro Devices, Inc.
  SPDX-License-Identifier: Apache-2.0
-->

# Hands-On Tutorial: amd-flashinfer on AMD Instinct GPUs

**amd-flashinfer** is the ROCm port of [FlashInfer](https://github.com/flashinfer-ai/flashinfer), a library of high-performance GPU operators for large language model (LLM) inference and serving. It provides fused attention, paged key–value cache layouts, sampling-related utilities, and related kernels adapted for AMD Instinct accelerators under the HIP programming model.

**Purpose of this tutorial**  
We demonstrate two practical pieces of an inference stack on ROCm: (1) **multi-head / grouped-query attention prefill**—the phase where query, key, and value tensors are combined to produce contextual representations for prompt tokens—and (2) the **`logits_processor`** pipeline, which applies temperature scaling, filtering (e.g. top-*k*), and sampling to raw logits from the final linear layer. Together, these illustrate how amd-flashinfer fits into a typical decode-time workflow alongside frameworks such as vLLM or SGLang.

**Target audience**  
Engineers and researchers who already use PyTorch on ROCm and want a concise, runnable introduction to amd-flashinfer APIs before integrating them into larger serving systems. Familiarity with attention mechanics (queries, keys, values, heads) is assumed.

**Scope**  
The exercises below use supported prefill and logits APIs in a single-node GPU environment. They do not cover every operator in the library. For broader feature coverage and installation matrices, see the [FlashInfer+ROCm README](https://github.com/ROCm/flashinfer/blob/main/README.md).

## Setting up amd-flashinfer

The [project README](https://github.com/ROCm/flashinfer/blob/main/README.md) describes supported ROCm and PyTorch versions (e.g. ROCm 7.0.2–7.2; Torch+ROCm 2.8.0 / 2.9.1) and **Instinct** architectures (**gfx942**, **gfx950**). Align your environment with one of the following before running the cells below.

**Option 1 — Pre-built Docker image (recommended for reproducibility)**  
AMD publishes images on Docker Hub under [`rocm/flashinfer`](https://hub.docker.com/r/rocm/flashinfer/tags). Start a container with GPU devices exposed, for example:

```bash
docker run -it --privileged --network=host --device=/dev/kfd --device=/dev/dri \
  --group-add video --cap-add=SYS_PTRACE --security-opt seccomp=unconfined \
  --ipc=host --shm-size 128G --name=<container-name> <docker-image-tag>
```

Inside the image, activate the provided environment (often `micromamba activate base`) and confirm `python -c "import flashinfer; print(flashinfer.__version__)"`.

**Option 2 — Wheel install on your own ROCm system**  
Install the package from AMD’s index and pair it with a **ROCm-enabled** PyTorch wheel from [repo.radeon.com](https://repo.radeon.com) (match major/minor ROCm to your stack; pin the torch version so a CPU-only wheel is not pulled from PyPI):

```bash
pip install amd-flashinfer --index-url https://pypi.amd.com/simple/
pip install torch==<version> -f https://repo.radeon.com/rocm/manylinux/rocm-rel-<rocm-version>
```

**Option 3 — Build from source**  
For development or custom builds, follow **Build from Source** in the README (development Dockerfile, editable install, or wheel build).

Once one of these paths is in place, continue with the next section to **verify** the interpreter you selected for this notebook.

## Verifying the runtime: PyTorch ROCm, FlashInfer, and GPU compatibility

The following cell imports amd-flashinfer and uses helpers from [`flashinfer/hip_utils.py`](https://github.com/ROCm/flashinfer/blob/main/flashinfer/hip_utils.py) to confirm that PyTorch was built with ROCm support, to report the detected ROCm version when possible, and to list GPU agents that match **FlashInfer-supported** architecture names (via `rocminfo`). It also checks that at least one accelerator is visible through the HIP runtime (`torch.cuda` on ROCm).

Run this cell **before** the worked examples; if it raises or warns, resolve the installation mismatch (README links above) before proceeding.

In [ ]:
import torch

import flashinfer
import flashinfer.hip_utils as hip_utils

device = "cuda:0"

hip_utils.check_torch_rocm_compatibility()

assert torch.cuda.is_available(), (
    "No HIP device visible to PyTorch. Check drivers, container device flags, "
    "and HIP_VISIBLE_DEVICES / CUDA_VISIBLE_DEVICES."
)

n_gpus = hip_utils.get_available_gpu_count()
assert n_gpus >= 1, f"Expected at least one visible GPU, got {n_gpus}"

print("flashinfer:", getattr(flashinfer, "__version__", "unknown"))
print("torch:", torch.__version__)
if hasattr(torch.version, "hip") and torch.version.hip:
    print("PyTorch HIP / ROCm build:", torch.version.hip)

rocm_ver = hip_utils.get_system_rocm_version()
print("Detected system ROCm version:", rocm_ver if rocm_ver else "(could not detect)")

print(
    "Architectures with AMD FlashInfer ports:",
    ", ".join(hip_utils.FLASHINFER_SUPPORTED_ROCM_ARCHS),
)

supported_idx = hip_utils.get_supported_device_indices()
print("GPU count (torch):", n_gpus)
print("Device indices FlashInfer treats as supported Instinct (rocminfo):", supported_idx)
if not supported_idx:
    print(
        "Note: rocminfo did not report a matching agent, or rocminfo is unavailable. "
        "You may still run on a supported GPU if the stack is correct."
    )

print("Using device:", device)

## Part A — Attention prefill

In transformer inference, **prefill** computes attention over prompt tokens so each position receives context from the rest of the sequence (subject to masking). amd-flashinfer exposes both **single-sequence** prefill and **batched prefill with a paged KV cache**, which is the layout many servers use to manage memory efficiently across layers and batches.

**Backend integration.** amd-flashinfer offers [AITER](https://github.com/ROCm/aiter)—AMD’s SOTA kernel library for large language model operators on ROCm—as an optional **`backend="aiter"`**. When the `aiter` package is present and the requested operator is integrated with AITER in this port, work **delegates** to AITER; otherwise the default HIP path applies. The following cells exercise the AITER-backed prefill path.

**Requirements and layout.** Install the `aiter` package (e.g. `amd-aiter` from AMD’s PyPI); see [AITER Support](https://github.com/ROCm/flashinfer/blob/main/README.md#aiter-support) in the README. These calls expect **NHD** key/value layout (`[sequence, heads, dim]`). For batched paged prefill, allowed **page sizes** depend on your AITER build (typical CK values are listed in the README, e.g. 1, 16, 1024).

In [ ]:
import importlib.util

if importlib.util.find_spec("aiter.ops") is None:
    raise RuntimeError(
        "The AITER Python package is required for the prefill examples in this tutorial.\n"
        "Install e.g.: pip install amd-aiter --index-url https://pypi.amd.com/simple/\n"
        "See README → AITER Support."
    )
print("AITER package available (aiter.ops).")

### Warm-up run

The first HIP kernel launch may trigger just-in-time compilation. Execute a small prefill once so later cells reflect steady-state timing if you benchmark.

In [ ]:
import math

dtype = torch.float16

_q = torch.randn(4, 8, 64, device=device, dtype=dtype)
_k = torch.randn(4, 4, 64, device=device, dtype=dtype)
_v = torch.randn(4, 4, 64, device=device, dtype=dtype)
_ = flashinfer.single_prefill_with_kv_cache(
    _q,
    _k,
    _v,
    causal=True,
    kv_layout="NHD",
    pos_encoding_mode="NONE",
    backend="aiter",
)
torch.cuda.synchronize()
print("Warm-up complete.")

### Single-sequence prefill

We use **grouped-query attention** (eight query heads, four KV heads): `k` and `v` are repeated logically along the head axis inside the kernel. **Causal** masking keeps each query position from attending to future keys. The call below sets `backend="aiter"` so prefill runs through AITER’s fused path; numerics are compared to a short PyTorch reference in float32.

In [ ]:
def naive_attention_prefill(q, k, v, causal: bool):
    qo_len, num_qo_heads, head_dim = q.shape
    kv_len, num_kv_heads, _ = k.shape
    sm_scale = 1.0 / math.sqrt(head_dim)
    g = num_qo_heads // num_kv_heads
    k = k.repeat_interleave(g, dim=1)
    v = v.repeat_interleave(g, dim=1)
    qt = q.transpose(0, 1)
    kt = k.transpose(0, 1)
    vt = v.transpose(0, 1)
    scores = torch.matmul(qt, kt.transpose(1, 2)) * sm_scale
    if causal:
        mask = torch.tril(
            torch.ones((qo_len, kv_len), device=q.device, dtype=torch.bool),
            diagonal=(kv_len - qo_len),
        )
        scores = scores.masked_fill(~mask.unsqueeze(0), float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return torch.matmul(attn, vt).transpose(0, 1)


qo_len, kv_len = 15, 127
num_qo_heads, num_kv_heads, head_dim = 8, 4, 64

q = torch.randn(qo_len, num_qo_heads, head_dim, device=device, dtype=dtype)
k = torch.randn(kv_len, num_kv_heads, head_dim, device=device, dtype=dtype)
v = torch.randn(kv_len, num_kv_heads, head_dim, device=device, dtype=dtype)

o = flashinfer.single_prefill_with_kv_cache(
    q,
    k,
    v,
    causal=True,
    kv_layout="NHD",
    pos_encoding_mode="NONE",
    backend="aiter",
)

o_ref = naive_attention_prefill(q.float(), k.float(), v.float(), causal=True).to(dtype)
torch.testing.assert_close(o, o_ref, rtol=1e-2, atol=1e-2)
print("Single-sequence prefill: OK — output shape", tuple(o.shape))

### Batched prefill with a paged KV cache

Serving systems often store keys and values in **fixed-size pages** and index them with indirection tables. The same logical batch can then be planned once and reused across layers. Here we build a minimal paged layout—`[total_pages, 2, page_size, num_kv_heads, head_dim]` for NHD, slot `0` for K and `1` for V—and run **`BatchPrefillWithPagedKVCacheWrapper`** with `backend="aiter"` in the constructor. We use `page_size=16` and a 512 MiB workspace buffer, matching the [batch prefill example](https://github.com/ROCm/flashinfer/blob/main/examples/batch_prefill_example.py). To build confidence, the first sequence in the batch is checked against the single-sequence prefill call above.

In [ ]:
batch_size = 4
kv_len = 128
qo_len = 128
page_size = 16
num_kv_heads = 4
num_qo_heads = 32
head_dim = 64
kv_layout = "NHD"

q = torch.randn(
    batch_size * qo_len, num_qo_heads, head_dim, device=device, dtype=dtype
)
q_indptr = torch.arange(0, batch_size + 1, device=device, dtype=torch.int32) * qo_len

num_pages_per_seq = (kv_len + page_size - 1) // page_size
total_num_pages = num_pages_per_seq * batch_size
kv_shape = [total_num_pages, 2, page_size, num_kv_heads, head_dim]
kv_data = torch.randn(*kv_shape, device=device, dtype=dtype)
kv_indptr = torch.arange(0, batch_size + 1, device=device, dtype=torch.int32) * num_pages_per_seq
kv_indices = torch.arange(0, total_num_pages, device=device, dtype=torch.int32)
kv_last_page_len = torch.full(
    (batch_size,), (kv_len - 1) % page_size + 1, dtype=torch.int32, device=device
)

workspace_buffer = torch.empty(512 * 1024 * 1024, dtype=torch.int8, device=device)
wrapper = flashinfer.prefill.BatchPrefillWithPagedKVCacheWrapper(
    workspace_buffer, kv_layout, backend="aiter"
)
wrapper.plan(
    q_indptr,
    kv_indptr,
    kv_indices,
    kv_last_page_len,
    num_qo_heads,
    num_kv_heads,
    head_dim,
    page_size,
    causal=False,
    pos_encoding_mode="NONE",
)

o_batch = wrapper.run(q, kv_data)
print("Batched paged prefill: output shape", tuple(o_batch.shape))


def reconstruct_seq_from_paged_nhd(kv_tensor, kv_ip, kv_lpl, seq_idx, kv_slot):
    chunks = []
    start = int(kv_ip[seq_idx].item())
    end = int(kv_ip[seq_idx + 1].item())
    last_tokens = int(kv_lpl[seq_idx].item())
    for p in range(start, end - 1):
        chunks.append(kv_tensor[p, kv_slot, :, :, :].reshape(-1, num_kv_heads, head_dim))
    p_last = end - 1
    chunks.append(
        kv_tensor[p_last, kv_slot, :last_tokens, :, :].reshape(-1, num_kv_heads, head_dim)
    )
    return torch.cat(chunks, dim=0)


k0 = reconstruct_seq_from_paged_nhd(kv_data, kv_indptr, kv_last_page_len, 0, 0)
v0 = reconstruct_seq_from_paged_nhd(kv_data, kv_indptr, kv_last_page_len, 0, 1)
assert k0.shape[0] == kv_len

q0 = q[:qo_len]
o0 = flashinfer.single_prefill_with_kv_cache(
    q0, k0, v0, causal=False, kv_layout="NHD", pos_encoding_mode="NONE", backend="aiter"
)
torch.testing.assert_close(o_batch[:qo_len], o0, rtol=1e-2, atol=1e-2)
print("Consistency check (batch vs single for sequence 0): OK")

## Part B — Logits processing after the last layer

After the model produces logits of shape `[batch, vocabulary]`, serving code often applies temperature, optional softmax, constrained sampling (top-*k*, top-*p*), and discrete **sampling**. amd-flashinfer exposes these steps as composable **`LogitsProcessor`** objects inside a **`LogitsPipe`**, which can be **compiled** for lower Python overhead. The snippet below uses random logits only to show the API; in production, logits would come from your transformer output projection.

This portion is independent of Part A and does not require AITER.

In [ ]:
from flashinfer.logits_processor import LogitsPipe, Softmax, Temperature, TopK, Sample

torch.manual_seed(0)
batch_size_lp, vocab_size = 32, 4096
logits = torch.randn(batch_size_lp, vocab_size, device=device, dtype=torch.float32)

pipe_eager = LogitsPipe([Temperature(), Softmax(), TopK(), Sample()], compile=False)
pipe_compiled = LogitsPipe([Temperature(), Softmax(), TopK(), Sample()], compile=True)

samples_eager = pipe_eager(logits, temperature=0.8, top_k=50)
samples_compiled = pipe_compiled(logits, temperature=0.8, top_k=50)

assert samples_eager.shape == (batch_size_lp,)
assert samples_compiled.shape == (batch_size_lp,)
assert (samples_eager >= 0).all() and (samples_eager < vocab_size).all()
assert (samples_compiled >= 0).all() and (samples_compiled < vocab_size).all()
print("LogitsPipe (eager vs compiled): sample indices in range — OK")

## Optional timing and further reading

If you measure latency, keep the warm-up from Part A in mind. **Decode-phase** attention (one new token per step against a growing cache) is available in amd-flashinfer’s HIP port; tuning it for production is typically done inside **serving frameworks** and complementary libraries. For MLA decode kernels on Instinct, see the [AITER MLA notebook](https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/notebooks/gpu_dev_optimize/aiter_mla_decode_kernel.html) linked earlier.

The cell below reports a rough wall-clock time for repeated single-sequence **AITER** prefill using the tensor sizes from Part A (not a rigorous benchmark).

In [ ]:
import time


def bench_single_prefill(n_warmup=3, n_iter=20):
    q = torch.randn(qo_len, num_qo_heads, head_dim, device=device, dtype=dtype)
    k = torch.randn(kv_len, num_kv_heads, head_dim, device=device, dtype=dtype)
    v = torch.randn(kv_len, num_kv_heads, head_dim, device=device, dtype=dtype)
    for _ in range(n_warmup):
        flashinfer.single_prefill_with_kv_cache(
            q, k, v, causal=True, kv_layout="NHD", pos_encoding_mode="NONE", backend="aiter"
        )
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        flashinfer.single_prefill_with_kv_cache(
            q, k, v, causal=True, kv_layout="NHD", pos_encoding_mode="NONE", backend="aiter"
        )
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / n_iter * 1000


ms = bench_single_prefill()
print(f"~{ms:.3f} ms per single-sequence prefill (AITER path), mean over iterations")

## Summary

This notebook introduced procedures to confirm a ROCm-enabled PyTorch stack with amd-flashinfer, including GPU and ROCm compatibility checks via `flashinfer.hip_utils`. It demonstrated **attention prefill** using the **AITER** backend for a single sequence and for **batched paged** key–value cache layouts, and it illustrated the **`LogitsPipe`** API for temperature scaling, filtering, and sampling on model logits. For supported platforms, container images, and the complete operator matrix, consult the [FlashInfer+ROCm README](https://github.com/ROCm/flashinfer/blob/main/README.md); for API details shared with upstream FlashInfer, see the [FlashInfer documentation](https://docs.flashinfer.ai).